[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%204/retrieval-hackathon/reference_solution.ipynb)

# Reference Solution — The Retrieval Hackathon

How the strongest retrievers are built, step by step, with the numbers each one reaches. It reads the same bundled data as the hackathon notebook — nothing to download. The progression is: keyword **baseline** → a **semantic** model → **retrieve-then-re-rank** (the ceiling).

## Setup — load the data and define scoring

Loads the 800-passage `corpus` and the two query sets (`main` = literal questions, `curve` = paraphrased) straight from the course repo. `scores()` is the exact metric the hackathon uses — **MRR@10** on each set, plus their average (`combined`). Every retriever below is scored with this one function, so the numbers compare directly.

In [ ]:
import json, urllib.request, numpy as np, time
BASE = "https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%204/retrieval-hackathon"
def load(n):
    try: raw = urllib.request.urlopen(f'{BASE}/{n}').read().decode()
    except Exception: raw = open(n).read()
    return [json.loads(l) for l in raw.splitlines() if l.strip()]
corpus=[r["text"] for r in sorted(load("corpus.jsonl"),key=lambda r:r["id"])]
main=load("queries.jsonl"); curve=load("curveball.jsonl")
def scores(rank):
    def mrr(qs):
        s=0
        for r in qs:
            o=rank(r["query"]); fr=next((j+1 for j,i in enumerate(o) if i==r["gold_id"]),999); s+=1/fr if fr<=10 else 0
        return s/len(qs)
    mn,cv=mrr(main),mrr(curve); return dict(main=mn, curve=cv, combined=(mn+cv)/2)
print("corpus",len(corpus),"| main",len(main),"| curveball",len(curve))
results={}

## 1 · TF-IDF baseline — the number to beat

The keyword baseline every team starts from. It turns each passage into a **sparse word-count vector** (weighted so rare words matter and common ones drop out) and ranks by cosine similarity to the query. Fast and interpretable — but it matches *words*, not *meaning*, so it does poorly on the paraphrased (`curve`) set. Watch that curveball number: **0.47**.

In [ ]:
# 1) TF-IDF baseline (what students start from)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
tf=TfidfVectorizer(stop_words="english"); X=tf.fit_transform(corpus)
def tfidf(q): return list(np.argsort(-cosine_similarity(tf.transform([q]),X)[0]))
results["TF-IDF baseline"]=scores(tfidf); print(results["TF-IDF baseline"])

## 2 · E5-small — semantic search (the strongest single model)

A **retrieval-tuned embedding model**: it encodes each passage and the query into dense *meaning* vectors and ranks by similarity, so paraphrases connect even with no shared words. The one detail that matters is the `"query: "` / `"passage: "` **prefixes** — E5 was trained to know which side is the question, and dropping them hurts. Notice the curveball jump vs. TF-IDF: **0.47 → 0.82**.

In [ ]:
# 2) E5-small (strongest single model -- note the query:/passage: prefixes)
!pip -q install sentence-transformers
from sentence_transformers import SentenceTransformer
e5=SentenceTransformer("intfloat/e5-small-v2")
De=e5.encode(["passage: "+d for d in corpus], normalize_embeddings=True)
def e5s(q): return list(np.argsort(-(De@e5.encode(["query: "+q],normalize_embeddings=True)[0])))
results["E5-small"]=scores(e5s); print(results["E5-small"])

## 3 · E5 + cross-encoder re-rank — the ceiling

The **two-stage retrieve-then-re-rank** pattern. E5 (a fast *bi-encoder*) pulls the top 50 candidates; then a **cross-encoder** — which reads the query and each passage *together* — re-scores just those 50 and reorders them. The cross-encoder is far more accurate but too slow to run over all 800, so it only ever sees the shortlist. This is the single biggest quality jump — and the slowest to run (~50 s).

In [ ]:
# 3) E5 + cross-encoder re-rank (THE CEILING) -- retrieve with E5, reorder top 50
from sentence_transformers import CrossEncoder
reranker=CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
def e5_rerank(q, topn=50):
    ids=e5s(q)[:topn]
    order=np.argsort(-reranker.predict([(q,corpus[i]) for i in ids]))
    return [ids[j] for j in order] + e5s(q)[topn:]
t=time.time()
results["E5 + rerank"]=scores(e5_rerank)
print(results["E5 + rerank"], f"({time.time()-t:.0f}s to score)")

## Results

In [ ]:
# Comparison table
print(f"{'method':<20}{'main':>7}{'curve':>7}{'combined':>10}")
for k,v in results.items():
    print(f"{k:<20}{v['main']:>7.3f}{v['curve']:>7.3f}{v['combined']:>10.3f}")

## Summary of results

| Method | Main MRR | Curveball MRR | **Combined** | Cost |
|---|---|---|---|---|
| TF-IDF baseline | 0.782 | 0.474 | **0.628** | instant |
| E5-small | 0.846 | 0.822 | **0.834** | ~seconds |
| **E5 + cross-encoder re-rank** | **0.937** | **0.868** | **0.902** | ~50 s |

**What the numbers say:**

- **Keyword is a strong baseline** on literal queries (0.782 main) but collapses on paraphrases (0.474) — it matches words, not meaning.
- **Semantic (E5) closes the paraphrase gap** — the curveball score nearly doubles (0.474 → 0.822) — and a good retrieval-tuned model beats a hybrid of weaker ones.
- **Re-ranking is the biggest single jump** (0.834 → 0.902 combined): retrieve fast, then re-score the top few precisely. It's also the most expensive, so it's a classic quality-vs-cost trade-off.

*ISBA 2411 · Natural Language Processing & AI · Summer 2026*